"""
Real-world data representation with tensors
PyTorch tensors, on the other hand, are homogeneous. Other data science packages,
such as Pandas, have the concept of the data frame, an object representing a data
set with named, heterogenous columns. By contrast, information in PyTorch is encoded
as a number, typically floating-point (though integer types are supported as well).
Numeric encoding is deliberate, because neural networks are mathematical entities that
take real numbers as inputs and produce real numbers as output through successive
application of matrix multiplications and nonlinear functions.

Your first job as a deep learning practitioner, therefore, is to encode heterogenous,
real-world data in a tensor of floating-point numbers, ready for consumption by a neural
network.
A large number of tabular data sets is freely available on the internet. See
https://github.com/caesar0301/awesome-public-data sets, for example.

We start with something fun: wine. The Wine Quality data set is a freely available
table containing chemical characterizations of samples of vinho verde (a wine from
northern Portugal) together with a sensory quality score. You can download the data set
for white wines at https://archive.ics.uci.edu/ml/machine-learning-databases/winequality/
winequality-white.csv. For convenience, we created a copy of the data set on the
Deep Learning with PyTorch Git repository, under data/p1ch4/tabular-wine.

Python offers several options for loading a CSV file quickly. Three popular options are
􀂃 The csv module that ships with Python
􀂃 NumPy
􀂃 Pandas

The third option is the most time- and memory-efficient, but we’ll avoid introducing
an additional library into your learning trajectory merely to load a file. Because we’ve
already introduced NumPy and PyTorch has excellent NumPy interoperability, you’ll
go with it. Load your file and turn the resulting NumPy array into a PyTorch tensor, as
shown in the following listing.
"""

In [1]:
import torch
import numpy as np
import csv

wine_path = "winequality-white.csv"
wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=";", skiprows=1)
wineq_numpy

array([[ 7.  ,  0.27,  0.36, ...,  0.45,  8.8 ,  6.  ],
       [ 6.3 ,  0.3 ,  0.34, ...,  0.49,  9.5 ,  6.  ],
       [ 8.1 ,  0.28,  0.4 , ...,  0.44, 10.1 ,  6.  ],
       ...,
       [ 6.5 ,  0.24,  0.19, ...,  0.46,  9.4 ,  6.  ],
       [ 5.5 ,  0.29,  0.3 , ...,  0.38, 12.8 ,  7.  ],
       [ 6.  ,  0.21,  0.38, ...,  0.32, 11.8 ,  6.  ]], dtype=float32)

In [2]:
# check that all the data has been read
col_list = next(csv.reader(open(wine_path), delimiter=';'))
wineq_numpy.shape, col_list

((4898, 12),
 ['fixed acidity',
  'volatile acidity',
  'citric acid',
  'residual sugar',
  'chlorides',
  'free sulfur dioxide',
  'total sulfur dioxide',
  'density',
  'pH',
  'sulphates',
  'alcohol',
  'quality'])

In [3]:
# convert the NumPy array to a PyTorch tensor:
wineq = torch.from_numpy(wineq_numpy)
wineq

tensor([[ 7.0000,  0.2700,  0.3600,  ...,  0.4500,  8.8000,  6.0000],
        [ 6.3000,  0.3000,  0.3400,  ...,  0.4900,  9.5000,  6.0000],
        [ 8.1000,  0.2800,  0.4000,  ...,  0.4400, 10.1000,  6.0000],
        ...,
        [ 6.5000,  0.2400,  0.1900,  ...,  0.4600,  9.4000,  6.0000],
        [ 5.5000,  0.2900,  0.3000,  ...,  0.3800, 12.8000,  7.0000],
        [ 6.0000,  0.2100,  0.3800,  ...,  0.3200, 11.8000,  6.0000]])

In [4]:
wineq.shape, wineq.type()

(torch.Size([4898, 12]), 'torch.FloatTensor')

In [5]:
data = wineq[:, :-1]
data.shape, data

(torch.Size([4898, 11]),
 tensor([[ 7.0000,  0.2700,  0.3600,  ...,  3.0000,  0.4500,  8.8000],
         [ 6.3000,  0.3000,  0.3400,  ...,  3.3000,  0.4900,  9.5000],
         [ 8.1000,  0.2800,  0.4000,  ...,  3.2600,  0.4400, 10.1000],
         ...,
         [ 6.5000,  0.2400,  0.1900,  ...,  2.9900,  0.4600,  9.4000],
         [ 5.5000,  0.2900,  0.3000,  ...,  3.3400,  0.3800, 12.8000],
         [ 6.0000,  0.2100,  0.3800,  ...,  3.2600,  0.3200, 11.8000]]))

In [6]:
# keep last column data seperately
target = wineq[:, -1]
target.shape, target

(torch.Size([4898]), tensor([6., 6., 6.,  ..., 6., 7., 6.]))

"""
If you want to transform the target tensor in a tensor of labels, you have two options,
depending on the strategy or how you want to use the categorical data. One option is
to treat a label as an integer vector of scores:
"""


In [7]:
target = wineq[:, -1].long()
target

tensor([6, 6, 6,  ..., 6, 7, 6])

In [8]:
type(target)

torch.Tensor

In [9]:
target.type()

'torch.LongTensor'

In [10]:
target.shape

torch.Size([4898])

"""
The other approach is to build a one-hot encoding of the scores—that is, encode
each of the ten scores in a vector of ten elements, with all elements set to zero but
one, at a different index for each score. This way, a score of 1 could be mapped to the
vector (1,0,0,0,0,0,0,0,0,0), a score of 5 to (0,0,0,0,1,0,0,0,0,0) and so on.
The fact that the score corresponds to the index of the nonzero element is purely incidental;
you could shuffle the assignment, and nothing would change from a classification
standpoint.

You can achieve one-hot encoding by using the scatter_ method, which fills the
tensor with values from a source tensor along the indices provided as arguments.
"""

In [11]:
target_onehot = torch.zeros(target.shape[0], 10)
target_onehot

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [12]:
target_onehot.shape

torch.Size([4898, 10])

In [13]:
target.unsqueeze(0)

tensor([[6, 6, 6,  ..., 6, 7, 6]])

In [14]:
target.unsqueeze(0).shape

torch.Size([1, 4898])

In [15]:
target.unsqueeze(1)

tensor([[6],
        [6],
        [6],
        ...,
        [6],
        [7],
        [6]])

In [16]:
target.unsqueeze(1).shape

torch.Size([4898, 1])

In [17]:
target

tensor([6, 6, 6,  ..., 6, 7, 6])

In [18]:
target.shape

torch.Size([4898])

tensor.scatter_(dim, index, src)

Parameters:
dim: The dimension along which to index. For example, dim=0 means updating rows, and dim=1 means updating columns.

index: A tensor containing the indices of the elements to update. The shape of index must match the shape of src.

src: The source tensor containing the values to scatter into the target tensor. It can also be a scalar value.

In [19]:
target_onehot.scatter_(1, target.unsqueeze(1), 1.0)

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [20]:
target_onehot[0]

tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])

In [21]:
target_onehot[-2], target[-2]

(tensor([0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]), tensor(7))

In [22]:
target_onehot[-2, 7]

tensor(1.)

"""
Now take a look at what scatter_ does. First, notice that its name ends with an underscore.
This convention in PyTorch indicates that the method won’t return a new tensor
but modify the tensor in place. The arguments for scatter_ are
􀂃 The dimension along which the following two arguments are specified
􀂃 A column tensor indicating the indices of the elements to scatter
􀂃 A tensor containing the elements to scatter or a single scalar to scatter (1, in
this case)

The second argument of scatter_, the index tensor, is required to have the same
number of dimensions as the tensor you scatter into. Because target_onehot has two
dimensions (4898x10), you need to add an extra dummy dimension to target by
using unsqueeze.

The call to unsqueeze adds a singleton dimension, from a 1D tensor of 4898 elements
to a 2D tensor of size (4898x1), without changing its contents. No elements were
added; you decided to use an extra index to access the elements. That is, you accessed
the first element of target as target[0] and the first element of its unsqueezed counterpart
as target_unsqueezed[0,0].

PyTorch allows you to use class indices directly as targets while training neural networks.
If you want to use the score as a categorical input to the network, however,
you’d have to transform it to a one-hot encoded tensor.
"""

In [23]:
data_mean= torch.mean(data, dim=0)
data_mean

tensor([6.8548e+00, 2.7824e-01, 3.3419e-01, 6.3914e+00, 4.5772e-02, 3.5308e+01,
        1.3836e+02, 9.9403e-01, 3.1883e+00, 4.8985e-01, 1.0514e+01])

In [24]:
data_mean.shape,  data_mean.type()

(torch.Size([11]), 'torch.FloatTensor')

data_mean2= torch.mean(data, dim=1)
data_mean2

tensor([23.3296, 15.3521, 14.3205,  ..., 14.8194, 14.0655, 13.0709])

data_mean2.shape
torch.Size([4898])

In [25]:
data_var = torch.var(data, dim=0)
data_var

tensor([7.1211e-01, 1.0160e-02, 1.4646e-02, 2.5726e+01, 4.7733e-04, 2.8924e+02,
        1.8061e+03, 8.9455e-06, 2.2801e-02, 1.3025e-02, 1.5144e+00])

In [26]:
data_var.shape, data_var.type()

(torch.Size([11]), 'torch.FloatTensor')

"""
In this case, dim=0 indicates that the reduction is performed along dimension 0. At
this point, you can normalize the data by subtracting the mean and dividing by the
standard deviation, which helps with the learning process.
"""

In [27]:
data_normalized = (data - data_mean) / torch.sqrt(data_var)
data_normalized

tensor([[ 1.7208e-01, -8.1761e-02,  2.1326e-01,  ..., -1.2468e+00,
         -3.4915e-01, -1.3930e+00],
        [-6.5743e-01,  2.1587e-01,  4.7996e-02,  ...,  7.3995e-01,
          1.3422e-03, -8.2419e-01],
        [ 1.4756e+00,  1.7450e-02,  5.4378e-01,  ...,  4.7505e-01,
         -4.3677e-01, -3.3663e-01],
        ...,
        [-4.2043e-01, -3.7940e-01, -1.1915e+00,  ..., -1.3130e+00,
         -2.6153e-01, -9.0545e-01],
        [-1.6054e+00,  1.1666e-01, -2.8253e-01,  ...,  1.0049e+00,
         -9.6251e-01,  1.8574e+00],
        [-1.0129e+00, -6.7703e-01,  3.7852e-01,  ...,  4.7505e-01,
         -1.4882e+00,  1.0448e+00]])

In [28]:
data_normalized.shape

torch.Size([4898, 11])

In [29]:
target.shape

torch.Size([4898])

"""
Next, look at the data with an eye to finding an easy way to tell good and bad wines
apart at a glance. First, use the torch.le function to determine which rows in target
correspond to a score less than or equal to 3:
"""

In [30]:
bad_indexes = torch.le(target, 3)
bad_indexes.shape, bad_indexes.dtype, bad_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(20))

In [31]:
bad_indexes

tensor([False, False, False,  ..., False, False, False])

In [32]:
data.shape

torch.Size([4898, 11])

"""
Note that only 20 of the bad_indexes entries are set to 1! By leveraging a feature in
PyTorch called advanced indexing, you can use a binary tensor to index the data tensor.
This tensor essentially filters data to be only items (or rows) that correspond to 1 in
the indexing tensor. The bad_indexes tensor has the same shape as target, with a
value of 0 or 1 depending on the outcome of the comparison between your threshold
and each element in the original target tensor:
"""

In [33]:
bad_data = data[bad_indexes]
bad_data

tensor([[8.5000e+00, 2.6000e-01, 2.1000e-01, 1.6200e+01, 7.4000e-02, 4.1000e+01,
         1.9700e+02, 9.9800e-01, 3.0200e+00, 5.0000e-01, 9.8000e+00],
        [5.8000e+00, 2.4000e-01, 4.4000e-01, 3.5000e+00, 2.9000e-02, 5.0000e+00,
         1.0900e+02, 9.9130e-01, 3.5300e+00, 4.3000e-01, 1.1700e+01],
        [9.1000e+00, 5.9000e-01, 3.8000e-01, 1.6000e+00, 6.6000e-02, 3.4000e+01,
         1.8200e+02, 9.9680e-01, 3.2300e+00, 3.8000e-01, 8.5000e+00],
        [7.1000e+00, 3.2000e-01, 3.2000e-01, 1.1000e+01, 3.8000e-02, 1.6000e+01,
         6.6000e+01, 9.9370e-01, 3.2400e+00, 4.0000e-01, 1.1500e+01],
        [6.9000e+00, 3.9000e-01, 4.0000e-01, 4.6000e+00, 2.2000e-02, 5.0000e+00,
         1.9000e+01, 9.9150e-01, 3.3100e+00, 3.7000e-01, 1.2600e+01],
        [1.0300e+01, 1.7000e-01, 4.7000e-01, 1.4000e+00, 3.7000e-02, 5.0000e+00,
         3.3000e+01, 9.9390e-01, 2.8900e+00, 2.8000e-01, 9.6000e+00],
        [7.9000e+00, 6.4000e-01, 4.6000e-01, 1.0600e+01, 2.4400e-01, 3.3000e+01,
         2.27

In [34]:
bad_data.shape

torch.Size([20, 11])

"""
Now you can start to get information about wines grouped into good, middling,
and bad categories. Take the .mean() of each column:
"""

In [35]:
bad_data = data[torch.le(target, 3)]
mid_data = data[torch.gt(target, 3) & torch.lt(target, 7)]
good_data = data[torch.gt(target, 7)]

bad_mean = torch.mean(bad_data, dim=0)
mid_mean = torch.mean(mid_data, dim=0)
good_mean = torch.mean(good_data, dim=0)

for i, args in enumerate(zip(col_list, bad_mean, mid_mean, good_mean)):
    print('{:2} {:20} {:6.2f} {:6.2f} {:6.2f}'.format(i, *args))

 0 fixed acidity          7.60   6.89   6.68
 1 volatile acidity       0.33   0.28   0.28
 2 citric acid            0.34   0.34   0.33
 3 residual sugar         6.39   6.71   5.63
 4 chlorides              0.05   0.05   0.04
 5 free sulfur dioxide   53.33  35.42  36.63
 6 total sulfur dioxide 170.60 141.83 125.88
 7 density                0.99   0.99   0.99
 8 pH                     3.19   3.18   3.22
 9 sulphates              0.47   0.49   0.49
10 alcohol               10.34  10.26  11.65


"""
It looks as though you’re on to something here. At first glance, the bad wines seem to
have higher total sulfur dioxide, among other differences. You could use a threshold
on total sulfur dioxide as a crude criterion for discriminating good wines from bad
ones. Now get the indexes in which the total sulfur dioxide column is below the midpoint
you calculated earlier, like so:
"""

In [36]:
total_sulfur_threshold = 141.83
total_sulfur_data = data[:, 6]
predicted_indexes = torch.lt(total_sulfur_data, total_sulfur_threshold)
predicted_indexes.shape, predicted_indexes.dtype, predicted_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(2727))

In [37]:
predicted_indexes

tensor([False,  True,  True,  ...,  True,  True,  True])

"""
Your threshold implies that slightly more than half of the wines are going to be high-quality.
Next, you need to get the indexes of the good wines:
"""

In [38]:
actual_indexes = torch.gt(target, 5)
actual_indexes.shape, actual_indexes.dtype, actual_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(3258))

In [39]:
actual_indexes

tensor([True, True, True,  ..., True, True, True])

"""
Because you have about 500 more good wines than your threshold predicted, you
already have hard evidence that the threshold isn’t perfect.
Now you need to see how well your predictions line up with the actual rankings.
Perform a logical AND between your prediction indexes and the good indexes
(remembering that each index is an array of 0s and 1s), and use that intersection of
wines in agreement to determine how well you did:

"""

In [40]:
n_matches = torch.sum(actual_indexes & predicted_indexes).item()
n_predicted = torch.sum(predicted_indexes).item()
n_actual = torch.sum(actual_indexes).item()

n_matches,  n_predicted,  n_actual

(2018, 2727, 3258)

In [41]:
n_matches, n_matches / n_predicted, n_matches / n_actual

(2018, 0.74000733406674, 0.6193984039287906)

"""
You got around 2,000 wines right! Because you had 2,700 wines predicted, a 74 percent
chance exists that if you predict a wine to be high-quality, it is. Unfortunately, you
have 3,200 good wines and identified only 61 percent of them. Well, we guess you got
what you signed up for; that result is barely better than random.


"""